In [0]:
# ============================================================================
# ETAPA 0A - PREPARAÇÃO DOS DADOS: REVIEWS (SILVER REFINADA)
# ============================================================================

from pyspark.sql.functions import (
    to_timestamp, date_format, hour, dayofweek, 
    col, when, current_timestamp
)

print("Preparando dados de Reviews...")
print("=" * 80)

# Carregar tabela de reviews do schema yelp_silver
df_reviews = spark.table("yelp_silver.review")

print(f"Total de reviews carregados: {df_reviews.count():,}")

# Criar colunas adicionais
df_reviews_refined = df_reviews \
    .withColumn("review_timestamp", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("review_date", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss").cast("date")) \
    .withColumn("review_hour", date_format(to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"), "HH:mm:ss")) \
    .withColumn("review_day_of_week_num", dayofweek(to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))) \
    .withColumn("review_day_of_week", 
        when(col("review_day_of_week_num") == 1, "Sunday")
        .when(col("review_day_of_week_num") == 2, "Monday")
        .when(col("review_day_of_week_num") == 3, "Tuesday")
        .when(col("review_day_of_week_num") == 4, "Wednesday")
        .when(col("review_day_of_week_num") == 5, "Thursday")
        .when(col("review_day_of_week_num") == 6, "Friday")
        .when(col("review_day_of_week_num") == 7, "Saturday")
    )

# Adicionar metadados de processamento
df_reviews_refined = df_reviews_refined.withColumn(
    "data_processamento_silver",
    current_timestamp()
)

print(f"\nColunas adicionadas:")
print("  - review_timestamp (timestamp)")
print("  - review_date (date)")
print("  - review_hour (string - formato HH:mm:ss)")
print("  - review_day_of_week (string)")
print("  - review_day_of_week_num (int)")

# Salvar como tabela Silver refinada
table_name = "yelp_ing.silver_reviews_refined"
df_reviews_refined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"\n✓ Tabela criada: {table_name}")
print(f"  Total de registros: {df_reviews_refined.count():,}")
print(f"  Total de colunas: {len(df_reviews_refined.columns)}")

# Mostrar amostra
print("\nAmostra dos dados:")
display(df_reviews_refined.select(
    "review_id", "business_id", "user_id", "date", 
    "review_timestamp", "review_date", "review_hour", "review_day_of_week"
).limit(5))

In [0]:
# ============================================================================
# ETAPA 1B - PREPARAÇÃO DOS DADOS: BUSINESS (SILVER REFINADA)
# ============================================================================

from pyspark.sql.functions import (
    col, explode, array, struct, lit, split, when, current_timestamp, 
    concat, lpad, to_timestamp, date_format, hour, minute
)

print("Preparando dados de Business...")
print("=" * 80)

# Carregar tabela de business do schema yelp_silver
df_business = spark.table("yelp_silver.business")

print(f"Total de estabelecimentos carregados: {df_business.count():,}")

# Explodir o campo 'hours' (struct) em linhas por dia da semana
df_business_exploded = df_business.select(
    "business_id",
    "name",
    "food_category",
    "city",
    "state",
    explode(array(
        struct(lit("Monday").alias("day"), col("hours.Monday").alias("hours_str")),
        struct(lit("Tuesday").alias("day"), col("hours.Tuesday").alias("hours_str")),
        struct(lit("Wednesday").alias("day"), col("hours.Wednesday").alias("hours_str")),
        struct(lit("Thursday").alias("day"), col("hours.Thursday").alias("hours_str")),
        struct(lit("Friday").alias("day"), col("hours.Friday").alias("hours_str")),
        struct(lit("Saturday").alias("day"), col("hours.Saturday").alias("hours_str")),
        struct(lit("Sunday").alias("day"), col("hours.Sunday").alias("hours_str"))
    )).alias("day_hours")
).select(
    "business_id",
    "name",
    "food_category",
    "city",
    "state",
    col("day_hours.day").alias("business_day_of_week"),
    col("day_hours.hours_str").alias("hours_str")
)

# Parsear o campo hours_str (formato: "11:0-22:0" ou "0:0-0:0")
# Criar timestamp auxiliar para comparações (data dummy 1970-01-01)
# E extrair apenas hora no formato HH:mm:ss para exibição
df_business_refined = df_business_exploded \
    .withColumn("open_time_ts", 
        when(col("hours_str").isNotNull(), 
            to_timestamp(
                concat(
                    lit("1970-01-01 "),
                    lpad(split(split(col("hours_str"), "-")[0], ":")[0], 2, "0"),
                    lit(":"),
                    lpad(split(split(col("hours_str"), "-")[0], ":")[1], 2, "0"),
                    lit(":00")
                )
            )
        )
    ) \
    .withColumn("close_time_ts", 
        when(col("hours_str").isNotNull(), 
            to_timestamp(
                concat(
                    lit("1970-01-01 "),
                    lpad(split(split(col("hours_str"), "-")[1], ":")[0], 2, "0"),
                    lit(":"),
                    lpad(split(split(col("hours_str"), "-")[1], ":")[1], 2, "0"),
                    lit(":00")
                )
            )
        )
    ) \
    .withColumn("open_time",
        when(col("open_time_ts").isNotNull(),
            date_format(col("open_time_ts"), "HH:mm:ss")
        )
    ) \
    .withColumn("close_time",
        when(col("close_time_ts").isNotNull(),
            date_format(col("close_time_ts"), "HH:mm:ss")
        )
    ) \
    .withColumn("is_closed_day",
        when(col("hours_str").isNull(), 0)
        .when(col("hours_str") == "0:0-0:0", 1)
        .otherwise(0)
    ) \
    .withColumn("data_processamento_gold", current_timestamp())

print(f"\nColunas adicionadas:")
print("  - business_day_of_week (string)")
print("  - open_time (string - formato HH:mm:ss)")
print("  - close_time (string - formato HH:mm:ss)")
print("  - open_time_ts (timestamp - para comparações internas)")
print("  - close_time_ts (timestamp - para comparações internas)")
print("  - is_closed_day (int - 1 se fechado naquele dia)")

# Salvar como tabela Silver refinada
table_name = "yelp_ing.silver_business_refined"
#df_business_refined.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela criada: {table_name}")
print(f"  Total de registros: {df_business_refined.count():,}")
print(f"  Total de colunas: {len(df_business_refined.columns)}")

# Mostrar amostra
print("\nAmostra dos dados:")
display(df_business_refined.select(
    "business_id", "name", "business_day_of_week", 
    "hours_str", "open_time", "close_time", "is_closed_day"
).limit(10))

In [0]:
# ============================================================================
# ETAPA 1 - CRUZAMENTO E DETECÇÃO DE POSSÍVEIS FRAUDES
# ============================================================================

from pyspark.sql.functions import col, when, concat, lit, current_timestamp, to_timestamp

print("Realizando cruzamento Reviews x Business e detectando possíveis anomalias...")
print("="  * 80)

# Carregar tabelas refinadas
df_reviews = df_reviews_refined
df_business = df_business_refined

print(f"Reviews carregados: {df_reviews.count():,}")
print(f"Business registros carregados: {df_business.count():,}")

# Join por business_id e dia da semana
df_joined = df_reviews.join(
    df_business,
    (df_reviews.business_id == df_business.business_id) &
    (df_reviews.review_day_of_week == df_business.business_day_of_week),
    "left"
)

print(f"\nRegistros após join: {df_joined.count():,}")

# Converter review_hour (que agora é HH:mm:ss) para timestamp (usando a mesma data dummy 1970-01-01)
df_joined = df_joined.withColumn(
    "review_time_timestamp",
    to_timestamp(
        concat(
            lit("1970-01-01 "),
            col("review_hour")
        )
    )
)

# LÓGICA DE DETECÇÃO DE POSSÍVEIS FRAUDES
# flag_possivel_fraude = 1 se:
# 1. Estabelecimento está permanentemente fechado (is_open = 0) - já tratado na camada silver
# 2. Review foi feita em dia que o estabelecimento não abre (is_closed_day = 1)
# 3. Review feita fora do horário de funcionamento

df_fraud_base = df_joined.withColumn(
    "flag_possivel_fraude",
    when(
        # Dia que não abre
        col("is_closed_day") == 1, 1
    ).when(
        # Fora do horário (review antes de abrir ou depois de fechar)
        # Usar open_time_ts e close_time_ts para comparações
        (col("open_time_ts").isNotNull()) & (col("close_time_ts").isNotNull()) &
        ((col("review_time_timestamp") < col("open_time_ts")) | (col("review_time_timestamp") > col("close_time_ts"))),
        0
    ).when(
        # Sem informação de horário (não podemos validar)
        col("open_time_ts").isNull(), 0
    ).otherwise(0)
)

# Selecionar colunas finais (usar open_time e close_time formatados para exibição)
df_fraud_base_final = df_fraud_base.select(
    df_reviews.business_id.alias("business_id"),
    df_reviews.review_id,
    df_reviews.user_id,
    df_reviews.stars,
    df_reviews.review_day_of_week,
    df_business.name.alias("business_name"),
    df_business.food_category,
    df_business.city,
    df_business.state,
    df_business.business_day_of_week,
    df_reviews.review_date,
    df_business.open_time,
    df_business.close_time,
    df_reviews.review_hour,
    df_reviews.review_timestamp,
    df_business.open_time_ts,
    df_business.close_time_ts,
    df_business.is_closed_day,
    "flag_possivel_fraude"
).withColumn("data_processamento_gold", current_timestamp())

# Estatísticas
total_reviews = df_fraud_base_final.count()
total_suspeitas = df_fraud_base_final.filter(col("flag_possivel_fraude") == 1).count()
total_valid = total_reviews - total_suspeitas

print(f"\n=" * 80)
print("ESTATÍSTICAS DE DETECÇÃO:")
print(f"  Total de reviews: {total_reviews:,}")
print(f"  Reviews suspeitas: {total_suspeitas:,} ({total_suspeitas/total_reviews*100:.2f}%)")
print(f"  Reviews válidas: {total_valid:,} ({total_valid/total_reviews*100:.2f}%)")
print(f"=" * 80)

# Salvar como tabela Gold
table_name = "yelp_ing.gold_review_fraud_base"
df_fraud_base_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar amostra de possíveis fraudes detectadas
print("\nAmostra de reviews com possíveis irregularidades:")
display(df_fraud_base_final.filter(col("flag_possivel_fraude") == 1).limit(100))

In [0]:
# ============================================================================
# ETAPA 3 - AGREGAÇÃO POR USUÁRIO E DATA
# ============================================================================

from pyspark.sql.functions import col, count, sum, when, concat, lit, current_timestamp

print("Agregando dados por usuário e data de review...")
print("=" * 80)

# Carregar base de detecção
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# Agregar por user_id e review_date
df_user_summary = df_fraud_base.groupBy("user_id", "review_date").agg(
    count("review_id").alias("total_reviews"),
    sum(when(col("flag_possivel_fraude") == 1, 1).otherwise(0)).alias("total_reviews_suspeitas")
)

# Criar mensagem de alerta
df_user_summary = df_user_summary.withColumn(
    "mensagem_alerta_user",
    when(
        col("total_reviews_suspeitas") > 0,
        concat(
            lit("Usuário "),
            col("user_id"),
            lit(" realizou "),
            col("total_reviews_suspeitas").cast("string"),
            lit(" review(s) em dias/horários em que o respectivo estabelecimento estava fechado")
        )
    ).otherwise(lit("Nenhuma irregularidade detectada"))
).withColumn("data_processamento_gold", current_timestamp())

# Estatísticas
total_records = df_user_summary.count()
records_with_issues = df_user_summary.filter(col("total_reviews_suspeitas") > 0).count()

print(f"\n=" * 80)
print("ESTATÍSTICAS POR USUÁRIO:")
print(f"  Total de registros (usuário + data): {total_records:,}")
print(f"  Registros com reviews suspeitas: {records_with_issues:,} ({records_with_issues/total_records*100:.2f}%)")
print(f"=" * 80)

# Dropar tabela se existir para evitar erro de schema mismatch
table_name = "yelp_ing.gold_user_fraud_summary"
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Salvar como tabela Gold
df_user_summary.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar usuários com mais reviews suspeitas
print("\nTop 10 registros (usuário + data) com mais reviews suspeitas:")
display(df_user_summary.filter(col("total_reviews_suspeitas") > 0).orderBy(col("total_reviews_suspeitas").desc()).limit(10))

In [0]:
# ============================================================================
# ETAPA 3 - AGREGAÇÃO POR USUÁRIO
# ============================================================================

from pyspark.sql.functions import col, count, sum, when, concat, lit, current_timestamp

print("Agregando dados por usuário...")
print("=" * 80)

# Carregar base de possível fraude
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# Agregar por user_id
df_user_summary = df_fraud_base.groupBy("user_id").agg(
    count("review_id").alias("total_reviews"),
    sum(when(col("flag_possivel_fraude") == 1, 1).otherwise(0)).alias("total_reviews_fraude")
)

# Criar mensagem de alerta
df_user_summary = df_user_summary.withColumn(
    "mensagem_alerta_user",
    when(
        col("total_reviews_fraude") > 0,
        concat(
            lit("User "),
            col("user_id"),
            lit(" realizou "),
            col("total_reviews_fraude").cast("string"),
            lit(" reviews em dias/horários em que o respectivo estabelecimento estava fechado")
        )
    ).otherwise(lit("Nenhuma possível fraude detectada"))
).withColumn("data_processamento_gold", current_timestamp())

# Estatísticas
total_users = df_user_summary.count()
users_with_fraud = df_user_summary.filter(col("total_reviews_fraude") > 0).count()

print(f"\n=" * 80)
print("ESTATÍSTICAS POR USUÁRIO:")
print(f"  Total de usuários: {total_users:,}")
print(f"  Usuários com possível fraude: {users_with_fraud:,} ({users_with_fraud/total_users*100:.2f}%)")
print(f"=" * 80)

# Dropar tabela se existir para evitar erro de schema mismatch
table_name = "yelp_ing.gold_user_fraud_summary"
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Salvar como tabela Gold
df_user_summary.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar usuários com mais possíveis fraudes
print("\nTop 10 usuários com mais reviews suspeitas:")
display(df_user_summary.filter(col("total_reviews_fraude") > 0).orderBy(col("total_reviews_fraude").desc()).limit(10))

In [0]:
# ============================================================================
# ETAPA 4 - AGREGAÇÃO POR ESTABELECIMENTO E DATA
# ============================================================================

from pyspark.sql.functions import (
    col, count, sum, when, concat, lit, countDistinct, 
    collect_set, array_join, current_timestamp
)

print("Agregando dados por estabelecimento e data de review...")
print("=" * 80)

# Carregar base de detecção
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# 1. Agregação básica por business_id, review_date
df_business_summary = df_fraud_base.groupBy("business_id", "business_name", "food_category", "review_date").agg(
    count("review_id").alias("total_reviews"),
    sum(when(col("flag_possivel_fraude") == 1, 1).otherwise(0)).alias("total_reviews_suspeitas")
)

# 2. Detectar comportamento suspeito por usuário
# Contar reviews por user_id, business_id e data
df_user_behavior = df_fraud_base.groupBy("user_id", "business_id", "review_date", "food_category").agg(
    count("review_id").alias("reviews_same_day_same_business")
)

# Contar reviews do mesmo usuário em restaurantes no mesmo dia
df_user_behavior_restaurants = df_fraud_base \
    .filter(col("food_category") == "RESTAURANTE") \
    .groupBy("user_id", "review_date").agg(
        countDistinct("business_id").alias("restaurants_reviewed_same_day")
    )

# Identificar alertas de comportamento suspeito
df_suspicious_users = df_user_behavior \
    .filter(col("reviews_same_day_same_business") > 10) \
    .select("user_id", "business_id", "review_date") \
    .distinct() \
    .withColumn(
        "alert_type",
        lit("same_business_many_reviews")
    )

df_suspicious_restaurants = df_user_behavior_restaurants \
    .filter(col("restaurants_reviewed_same_day") > 5) \
    .join(df_fraud_base.select("user_id", "business_id", "review_date").distinct(), ["user_id", "review_date"]) \
    .select("user_id", "business_id", "review_date") \
    .distinct() \
    .withColumn(
        "alert_type",
        lit("many_restaurants_same_day")
    )

# Unir alertas
df_all_alerts = df_suspicious_users.union(df_suspicious_restaurants)

# Agregar alertas por business e data
df_alerts_by_business = df_all_alerts.groupBy("business_id", "review_date").agg(
    collect_set(
        when(col("alert_type") == "same_business_many_reviews",
            concat(
                lit("Usuário "),
                col("user_id"),
                lit(" realizou mais de 10 reviews no mesmo estabelecimento no mesmo dia")
            )
        ).when(col("alert_type") == "many_restaurants_same_day",
            concat(
                lit("Usuário "),
                col("user_id"),
                lit(" realizou mais de 5 reviews em Restaurantes no mesmo dia")
            )
        )
    ).alias("user_alerts_list")
)

# Join com o sumário de business
df_business_final = df_business_summary.join(
    df_alerts_by_business,
    ["business_id", "review_date"],
    "left"
)

# Criar mensagens de alerta
df_business_final = df_business_final \
    .withColumn(
        "mensagem_alerta_business",
        when(
            col("total_reviews_suspeitas") > 0,
            concat(
                lit("Estabelecimento "),
                col("business_name"),
                lit(" recebeu "),
                col("total_reviews_suspeitas").cast("string"),
                lit(" review(s) em dias/horários em que estava fechado")
            )
        ).otherwise(lit("Nenhuma irregularidade detectada"))
    ) \
    .withColumn(
        "mensagem_alerta_user_2",
        when(
            col("user_alerts_list").isNotNull(),
            array_join(col("user_alerts_list"), "; ")
        ).otherwise(lit("Nenhum comportamento suspeito detectado"))
    ) \
    .withColumn("data_processamento_gold", current_timestamp()) \
    .drop("user_alerts_list")

# Estatísticas
total_records = df_business_final.count()
records_with_issues = df_business_final.filter(col("total_reviews_suspeitas") > 0).count()

print(f"\n=" * 80)
print("ESTATÍSTICAS POR ESTABELECIMENTO:")
print(f"  Total de registros (estabelecimento + data): {total_records:,}")
print(f"  Registros com reviews suspeitas: {records_with_issues:,} ({records_with_issues/total_records*100:.2f}%)")
print(f"=" * 80)

# Salvar como tabela Gold com overwriteSchema=True
table_name = "yelp_ing.gold_business_fraud_summary"
df_business_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar estabelecimentos com mais reviews suspeitas
print("\nTop 10 registros (estabelecimento + data) com mais reviews suspeitas:")
display(df_business_final.filter(col("total_reviews_suspeitas") > 0).orderBy(col("total_reviews_suspeitas").desc()).limit(10))

In [0]:
# ============================================================================
# ETAPA 5 - MÉTRICAS GERAIS POR DATA
# ============================================================================

from pyspark.sql.functions import col, count, sum, when, lit, current_timestamp

print("Calculando métricas gerais por data de review...")
print("=" * 80)

# Carregar base de detecção
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# Calcular métricas agrupadas por review_date
df_metrics = df_fraud_base.groupBy("review_date").agg(
    count("review_id").alias("total_reviews"),
    sum(when(col("flag_possivel_fraude") == 1, 1).otherwise(0)).alias("total_reviews_suspeitas"),
    sum(when(col("flag_possivel_fraude") == 0, 1).otherwise(0)).alias("total_reviews_validas")
).withColumn("data_processamento_gold", current_timestamp())

# Estatísticas gerais
total_reviews = df_fraud_base.count()
total_suspeitas = df_fraud_base.filter(col("flag_possivel_fraude") == 1).count()
total_valid = total_reviews - total_suspeitas

print(f"\n=" * 80)
print("MÉTRICAS GERAIS CONSOLIDADAS:")
print(f"  Total de reviews: {total_reviews:,}")
print(f"  Reviews suspeitas: {total_suspeitas:,} ({total_suspeitas/total_reviews*100:.2f}%)")
print(f"  Reviews válidas: {total_valid:,} ({total_valid/total_reviews*100:.2f}%)")
print(f"=" * 80)

# Salvar como tabela Gold
table_name = "yelp_ing.gold_fraud_metrics"
df_metrics.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")
print(f"  Registros: uma linha por data de review")

# Mostrar métricas
print("\nAmostra das métricas por data:")
display(df_metrics.orderBy(col("review_date").desc()).limit(10))

In [0]:
# ============================================================================
# ETAPA 6 - DISTRIBUIÇÃO TEMPORAL POR DATA E FAIXA DE HORÁRIO
# ============================================================================

from pyspark.sql.functions import col, when, count, sum, current_timestamp, substring

print("Calculando distribuição temporal por data e faixa de horário...")
print("=" * 80)

# Carregar base de detecção
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

# Extrair a hora como inteiro do review_hour (formato "HH:MM:SS")
df_with_hour = df_fraud_base.withColumn(
    "hour_int",
    substring(col("review_hour"), 1, 2).cast("int")
)

# Criar buckets de horário
df_time_distribution = df_with_hour.withColumn(
    "time_bucket",
    when((col("hour_int") >= 0) & (col("hour_int") < 3), "00:00-03:00")
    .when((col("hour_int") >= 3) & (col("hour_int") < 6), "03:01-06:00")
    .when((col("hour_int") >= 6) & (col("hour_int") < 9), "06:01-09:00")
    .when((col("hour_int") >= 9) & (col("hour_int") < 12), "09:01-12:00")
    .when((col("hour_int") >= 12) & (col("hour_int") < 15), "12:01-15:00")
    .when((col("hour_int") >= 15) & (col("hour_int") < 18), "15:01-18:00")
    .when((col("hour_int") >= 18) & (col("hour_int") < 21), "18:01-21:00")
    .when((col("hour_int") >= 21) & (col("hour_int") <= 23), "21:01-23:59")
    .otherwise("Unknown")
)

# Agregar por review_date e time_bucket
df_time_agg = df_time_distribution.groupBy("review_date", "time_bucket").agg(
    count("review_id").alias("total_reviews"),
    sum(when(col("flag_possivel_fraude") == 1, 1).otherwise(0)).alias("total_suspeitas"),
    sum(when(col("flag_possivel_fraude") == 0, 1).otherwise(0)).alias("total_validas")
).withColumn("data_processamento_gold", current_timestamp())

# Ordenar por data e horário
df_time_agg = df_time_agg.orderBy(
    col("review_date"),
    when(col("time_bucket") == "00:00-03:00", 1)
    .when(col("time_bucket") == "03:01-06:00", 2)
    .when(col("time_bucket") == "06:01-09:00", 3)
    .when(col("time_bucket") == "09:01-12:00", 4)
    .when(col("time_bucket") == "12:01-15:00", 5)
    .when(col("time_bucket") == "15:01-18:00", 6)
    .when(col("time_bucket") == "18:01-21:00", 7)
    .when(col("time_bucket") == "21:01-23:59", 8)
)

print(f"\nTotal de registros: {df_time_agg.count():,}")
print("  (Uma linha por combinação de data + faixa horária)")

# Salvar como tabela Gold
table_name = "yelp_ing.gold_reviews_time_distribution"
df_time_agg.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"\n✓ Tabela Gold criada: {table_name}")

# Mostrar amostra
print("\nAmostra da distribuição temporal:")
display(df_time_agg.limit(20))

In [0]:
# ============================================================================
# ETAPA 7 - QUERIES PARA DASHBOARD
# ============================================================================

print("=" * 80)
print("QUERIES SQL PRONTAS PARA DASHBOARD")
print("=" * 80)

queries = {
    "KPI - Métricas Gerais (filtrado por data)": """
        SELECT 
            review_date,
            SUM(total_reviews) as total_reviews,
            SUM(total_reviews_suspeitas) as total_reviews_suspeitas,
            SUM(total_reviews_validas) as total_reviews_validas,
            ROUND(SUM(total_reviews_suspeitas) * 100.0 / SUM(total_reviews), 2) as percentual_suspeitas
        FROM yelp_ing.gold_fraud_metrics
        GROUP BY review_date
    """,
    
    "Distribuição Temporal (Gráfico de Barras)": """
        SELECT 
            review_date,
            time_bucket,
            SUM(total_reviews) as total_reviews,
            SUM(total_suspeitas) as total_suspeitas,
            SUM(total_validas) as total_validas
        FROM yelp_ing.gold_reviews_time_distribution
        GROUP BY review_date, time_bucket
        ORDER BY 
            review_date,
            CASE time_bucket
                WHEN '00:00-03:00' THEN 1
                WHEN '03:01-06:00' THEN 2
                WHEN '06:01-09:00' THEN 3
                WHEN '09:01-12:00' THEN 4
                WHEN '12:01-15:00' THEN 5
                WHEN '15:01-18:00' THEN 6
                WHEN '18:01-21:00' THEN 7
                WHEN '21:01-23:59' THEN 8
            END
    """,
    
    "Alertas de Usuários (Top 20 por data)": """
        SELECT 
            review_date,
            user_id,
            total_reviews,
            total_reviews_suspeitas,
            mensagem_alerta_user
        FROM yelp_ing.gold_user_fraud_summary
        WHERE total_reviews_suspeitas > 0
        ORDER BY review_date DESC, total_reviews_suspeitas DESC
        LIMIT 20
    """,
    
    "Alertas de Estabelecimentos (Top 20 por data)": """
        SELECT 
            review_date,
            business_name,
            food_category,
            total_reviews,
            total_reviews_suspeitas,
            mensagem_alerta_business,
            mensagem_alerta_user_2
        FROM yelp_ing.gold_business_fraud_summary
        WHERE total_reviews_suspeitas > 0
        ORDER BY review_date DESC, total_reviews_suspeitas DESC
        LIMIT 20
    """,
    
    "Reviews com Possíveis Irregularidades (Detalhadas)": """
        SELECT 
            review_id,
            user_id,
            business_name,
            food_category,
            review_date,
            review_hour,
            review_day_of_week,
            is_open,
            open_time,
            close_time,
            flag_possivel_fraude
        FROM yelp_ing.gold_review_fraud_base
        WHERE flag_possivel_fraude = 1
        ORDER BY review_date DESC
        LIMIT 100
    """
}

for query_name, query_sql in queries.items():
    print(f"\n{'=' * 80}")
    print(f"QUERY: {query_name}")
    print(f"{'=' * 80}")
    print(query_sql.strip())
    print()

print("\n" + "=" * 80)
print("RESUMO DAS TABELAS CRIADAS")
print("=" * 80)

tabelas_criadas = [
    ("yelp_ing.silver_reviews_refined", "Reviews com colunas de data/hora extraídas"),
    ("yelp_ing.silver_business_refined", "Business com horários explodidos por dia da semana"),
    ("yelp_ing.gold_review_fraud_base", "Base completa com flag de possíveis fraudes"),
    ("yelp_ing.gold_user_fraud_summary", "Agregação por usuário e data com alertas"),
    ("yelp_ing.gold_business_fraud_summary", "Agregação por estabelecimento e data com alertas"),
    ("yelp_ing.gold_fraud_metrics", "Métricas gerais por data (KPIs)"),
    ("yelp_ing.gold_reviews_time_distribution", "Distribuição temporal por data e faixa de horário")
]

print("\nTabelas Silver Refinada:")
for tabela, descricao in tabelas_criadas[:2]:
    print(f"  ✓ {tabela}")
    print(f"    {descricao}")

print("\nTabelas Gold:")
for tabela, descricao in tabelas_criadas[2:]:
    print(f"  ✓ {tabela}")
    print(f"    {descricao}")

print("\n" + "=" * 80)
print("✓ PIPELINE COMPLETO CRIADO COM SUCESSO!")
print("=" * 80)
print("\nUse as queries SQL acima para criar visualizações no Databricks SQL.")
print("Todas as tabelas incluem review_date para filtragem no dashboard.")
print("\n")

In [0]:
# ============================================================================
# RESUMO EXECUTIVO - PIPELINE DE DETEÇÃO DE POSSÍVEIS IRREGULARIDADES YELP
# ============================================================================

from pyspark.sql.functions import col

print("\n" + "=" * 80)
print("RESUMO EXECUTIVO - PIPELINE DE DETEÇÃO DE POSSÍVEIS IRREGULARIDADES")
print("=" * 80)

print("\n📊 ESTATÍSTICAS CONSOLIDADAS:")
print("-" * 80)

# Carregar métricas consolidadas
df_fraud_base = spark.table("yelp_ing.gold_review_fraud_base")

total_reviews = df_fraud_base.count()
total_suspeitas = df_fraud_base.filter(col("flag_possivel_fraude") == 1).count()
total_validas = total_reviews - total_suspeitas

print(f"  Total de Reviews Analisados: {total_reviews:,}")
print(f"  Reviews Suspeitas: {total_suspeitas:,} ({total_suspeitas/total_reviews*100:.2f}%)")
print(f"  Reviews Válidas: {total_validas:,} ({total_validas/total_reviews*100:.2f}%)")

# Estatísticas por usuário
df_users = spark.table("yelp_ing.gold_user_fraud_summary")
total_user_records = df_users.count()
user_records_with_issues = df_users.filter(col("total_reviews_fraude") > 0).count()

print(f"\n  Total de Registros Usuário+Data: {total_user_records:,}")
print(f"  Registros com Reviews Suspeitas: {user_records_with_issues:,} ({user_records_with_issues/total_user_records*100:.2f}%)")

# Estatísticas por estabelecimento
df_business = spark.table("yelp_ing.gold_business_fraud_summary")
total_business_records = df_business.count()
business_records_with_issues = df_business.filter(col("total_reviews_suspeitas") > 0).count()

print(f"\n  Total de Registros Estabelecimento+Data: {total_business_records:,}")
print(f"  Registros com Reviews Suspeitas: {business_records_with_issues:,} ({business_records_with_issues/total_business_records*100:.2f}%)")

# Distribuição temporal
print("\n🕒 HORÁRIOS COM MAIS REVIEWS SUSPEITAS:")
print("-" * 80)
df_time = spark.table("yelp_ing.gold_reviews_time_distribution")

# Agregar por time_bucket (consolidando todas as datas)
df_time_consolidated = df_time.groupBy("time_bucket").agg(
    sum(col("total_reviews")).alias("total_reviews"),
    sum(col("total_suspeitas")).alias("total_suspeitas")
)

top_suspect_hours = df_time_consolidated.filter(col("total_suspeitas") > 0).orderBy(col("total_suspeitas").desc()).limit(3).collect()

for i, row in enumerate(top_suspect_hours, 1):
    if row['total_reviews'] > 0:
        print(f"  {i}. {row['time_bucket']}: {row['total_suspeitas']} suspeitas de {row['total_reviews']} reviews ({row['total_suspeitas']/row['total_reviews']*100:.2f}%)")

print("\n" + "=" * 80)
print("🛠️ ARQUITETURA IMPLEMENTADA:")
print("=" * 80)
print("\n  Camadas de Dados:")
print("    ✓ Bronze (já existente): Dados brutos do Yelp")
print("    ✓ Silver: Tabelas de amostra (1000 registros)")
print("    ✓ Silver Refinada: Dados enriquecidos com informações temporais")
print("    ✓ Gold: Tabelas analíticas com dimensão de data para filtragem")

print("\n  Lógica de Detecção:")
print("    ✓ Estabelecimento permanentemente fechado (is_open = 0)")
print("    ✓ Review em dia que estabelecimento não abre")
print("    ✓ Review fora do horário de funcionamento")

print("\n  Filtragem por Data:")
print("    ✓ Todas as tabelas Gold incluem review_date")
print("    ✓ Dashboard pode filtrar por data específica")
print("    ✓ Métricas e gráficos respondem ao filtro de data")

print("\n" + "=" * 80)
print("🚀 PRÓXIMOS PASSOS PARA ESCALAR:")
print("=" * 80)
print("\n  1. Trocar tabelas de amostra pelas tabelas completas:")
print("     - yelp_ing.silver_reviews_filtered_amostra → yelp_ing.silver_review_filtered")
print("     - yelp_ing.silver_business_amostra → yelp_ing.silver_business")

print("\n  2. Ajustar particionamento para performance:")
print("     - Particionar por review_date na camada Gold")
print("     - Adicionar Z-ORDER em colunas de filtro frequente")

print("\n  3. Adicionar detecções avançadas:")
print("     - Padrões de texto suspeitos (reviews genéricas)")
print("     - Velocidade de publicação (muitos reviews em curto período)")
print("     - Análise de localização geográfica")

print("\n  4. Automatizar com Lakeflow Spark Declarative Pipelines:")
print("     - Pipeline incremental")
print("     - Expectativas de qualidade de dados")
print("     - Monitoramento contínuo")

print("\n  5. Dashboard Databricks:")
print("     - Filtro de data (review_date)")
print("     - KPI Cards com métricas principais")
print("     - Gráficos de distribuição temporal")
print("     - Alertas de usuários e estabelecimentos")

print("\n" + "=" * 80)
print("✅ PIPELINE PRONTO PARA DASHBOARD!")
print("=" * 80)
print("\nUse as queries SQL da ETAPA 7 para criar visualizações.")
print("Todas incluem review_date para filtragem dinâmica.")
print("\n")

In [0]:
from pyspark.sql.functions import col

# Filtrar a review específica na tabela silver_reviews_refined
df_review_filtrado = spark.table("yelp_silver.review").filter((col("user_id") == "ftIPVahZ98qFdtxdOoQGmA") & (col("business_id") == "PP3BBaVxZLcJU54uP_wL6Q"))
display(df_review_filtrado)

# Filtrar o estabelecimento específico na tabela silver_business_refined
df_business_filtrado = spark.table("yelp_silver.business").filter(col("business_id") == "PP3BBaVxZLcJU54uP_wL6Q")
display(df_business_filtrado)